In [1]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [4]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from typing import Literal

class SentimentOutputSchema(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(..., description="The sentiment of the review.")

sentiment_llm = llm.with_structured_output(SentimentOutputSchema)

In [5]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal["positive", "negative"]
    assessment: dict
    response: str

In [6]:
from langchain_core.prompts import PromptTemplate

def find_sentiment(ReviewState):
    review = ReviewState['review']
    prompt_template = PromptTemplate(
        input_variables=["review"],
        template="Determine the sentiment of the following review: {review}. Respond with either 'positive' or 'negative'."
    )
    chain = prompt_template | sentiment_llm
    sentiment_result = chain.invoke({"review": review})
    return {"sentiment" : sentiment_result.sentiment}

In [7]:
graph = StateGraph(ReviewState)

In [8]:
def check_sentiment_mood(ReviewState):
    sentiment = ReviewState['sentiment']
    if sentiment == "positive":
        return 'generate_positive_response'
    else:
        return 'run_assessment'

In [9]:
class AssessmentOutputSchema(BaseModel):
    mood: Literal['angry', 'disappointed', 'constructive', 'calm'] = Field(..., description="The mood/tonality of the review, either 'angry', 'disappointed', or 'constructive', 'calm'.")
    issue_type: Literal['product', 'service', 'delivery', 'bug'] = Field(..., description="The type of issue mentioned in the review, either 'product', 'service', 'delivery', or 'bug'.")
    urgency: Literal['high', 'medium', 'low'] = Field(..., description="The urgency of the issue mentioned in the review, either 'high', 'medium', or 'low'.")

In [10]:
assessement_llm = llm.with_structured_output(AssessmentOutputSchema)

In [12]:
from langchain_core.output_parsers import StrOutputParser

def run_assessment(ReviewState):
    review = ReviewState['review']
    prompt_template = PromptTemplate(
        input_variables=["review"],
        template="Assess the following review: \n\n{review}. \n\nProvide a detailed analysis of the issues mentioned."
    )
    chain = prompt_template | assessement_llm
    assessment_result = chain.invoke({"review": review})
    return {"assessment": assessment_result.model_dump()}

def generate_positive_response(ReviewState):
    review = ReviewState['review']
    prompt_template = PromptTemplate(
        input_variables=["review"],
        template="Generate a positive thank you message to the following review: \n{review}."
    )
    parser = StrOutputParser()
    chain = prompt_template | llm | parser
    response_result = chain.invoke({"review": review})
    return {"response": response_result}

def generate_negative_response(ReviewState):
    review = ReviewState['review']
    assessment = ReviewState['assessment']
    prompt_template = PromptTemplate(
        input_variables=["review", "assessment"],
        template="Generate a professional empathetic response to the following review: \n{review}. \n\nThe assessment of the review is as follows: {assessment}. \n\nPlease address the issues mentioned in the review and provide a solution or next steps."
    )
    parser = StrOutputParser()
    chain = prompt_template | llm | parser
    response_result = chain.invoke({"review": review, "assessment": assessment})
    return {"response": response_result}

In [13]:
graph.add_node('find_sentiment', find_sentiment)
graph.add_node('run_assessment', run_assessment)
graph.add_node('generate_positive_response', generate_positive_response)
graph.add_node('generate_negative_response', generate_negative_response)

In [14]:
graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges('find_sentiment', check_sentiment_mood)
graph.add_edge('run_assessment', 'generate_negative_response')
graph.add_edge('generate_positive_response', END)
graph.add_edge('generate_negative_response', END)

In [16]:
workflow = graph.compile()

In [17]:
input_state = {
    "review" : "I recently purchased a product from your store, and I am extremely disappointed with the quality. The item arrived damaged, and the customer service was unhelpful when I tried to resolve the issue. I expected better from your company."
}

In [18]:
final_state = workflow.invoke(input_state)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


In [19]:
print("Response:", final_state['response'])

Response: Dear [Customer Name],

Thank you for reaching out and sharing your feedback with us. I am truly sorry to learn about your recent experience, both with the damaged item you received and the lack of support from our customer service team. This is certainly not the standard of quality or care we strive to provide, and I completely understand your disappointment. 

We take issues like this very seriously. To make things right immediately, we would like to offer you a full refund or a replacement of the item shipped via expedited delivery at no additional cost—whichever you prefer. 

To help us resolve this for you as quickly as possible, could you please reply to this message or contact me directly at [Email Address/Phone Number] with your order number? Additionally, if you are able to share a quick photo of the damaged item, it would help us investigate the issue with our warehouse and shipping partners.

We deeply value your business and hope to have the opportunity to restore 

In [20]:
print(final_state)

{'review': 'I recently purchased a product from your store, and I am extremely disappointed with the quality. The item arrived damaged, and the customer service was unhelpful when I tried to resolve the issue. I expected better from your company.', 'sentiment': 'negative', 'assessment': {'mood': 'disappointed', 'issue_type': 'product', 'urgency': 'medium'}, 'response': 'Dear [Customer Name],\n\nThank you for reaching out and sharing your feedback with us. I am truly sorry to learn about your recent experience, both with the damaged item you received and the lack of support from our customer service team. This is certainly not the standard of quality or care we strive to provide, and I completely understand your disappointment. \n\nWe take issues like this very seriously. To make things right immediately, we would like to offer you a full refund or a replacement of the item shipped via expedited delivery at no additional cost—whichever you prefer. \n\nTo help us resolve this for you as 

In [21]:
input_state_2 = {
    "review" : "Your website is very user-friendly and I had a great experience navigating through it. The product descriptions were clear and the checkout process was smooth. I will definitely recommend your store to my friends and family."
}

In [22]:
final_state_2 = workflow.invoke(input_state_2)

In [23]:
print("Response: ", final_state_2['response'])

Response:  Here is a warm and positive response you can use:

"Thank you so much for the wonderful feedback! We are thrilled to hear that you had such a smooth and easy experience navigating our site and checking out. Providing a user-friendly shopping experience is a top priority for us, and your kind words mean the world to our team. We truly appreciate your support and look forward to welcoming you—and your friends and family—back soon!"
